# Automatizando bot

### Passo a Passo:
- Pegar link do jogo
- Abrir link    (exemplo: https://www.vlr.gg/596425/100-thieves-vs-cloud9-vct-2026-americas-kickoff-lr4)
- Encontrar times (class: wf-title-med)
- Placar (classe: js-spoiler)
- PickBan (classe: match-header-note)
- Encontrar mapas (classe: js-map-switch)
- Encontrar agentes (classe: mod-agets; tag: img; .title)
- Encontrar rounds (classe: vlr-rounds-row-col .title)
- Mapa acabou? classe mod-win aparece

In [ ]:
# Inicializando Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
import os

navegador = webdriver.Chrome()
link = 'https://www.vlr.gg/598922/jdg-esports-vs-titan-esports-club-vct-2026-china-kickoff-ur1'

In [ ]:
# Abrindo o site
navegador.get(link)

#### Searching Stats


In [ ]:
nav_itens = navegador.find_elements(By.CLASS_NAME, 'wf-nav-item')
for item in nav_itens:
    if 'stats' in item.get_attribute('href'):
        item.click()
        link = navegador.current_url
        break

In [ ]:
import pandas as pd

In [ ]:
stats_table = navegador.find_element(By.TAG_NAME, "table")

columns = stats_table.find_element(By.TAG_NAME, "thead") \
                     .find_elements(By.TAG_NAME, "th")

columns = [column.text for column in columns]

In [ ]:
rows = stats_table.find_element(By.TAG_NAME, "tbody") \
                  .find_elements(By.TAG_NAME, "tr")

table = []
for row in rows:
    stats = row.find_elements(By.TAG_NAME, "td")
    stats = [stat.text for stat in stats]
    table.append(stats)

In [ ]:
tabela = pd.DataFrame(table, columns=columns)

display(tabela.head())
display(tabela.info())

In [ ]:
tabela = tabela.drop(columns=["AGENTS", "RND", "KMAX", "K", "D", "A", "FK", "FD", "CL%"])
display(tabela.head())
display(tabela.info())

In [ ]:
new_column = {"R2.0": "RATING", "K:D": "K/D"}
tabela = tabela.rename(columns=new_column)
display(tabela.head())

In [ ]:
tabela["TEAM"] = "N/A"
for linha in tabela.index:
    player, time = tabela.loc[linha, "PLAYER"].split("\n")
    tabela.loc[linha, "PLAYER"] = player
    tabela.loc[linha, "TEAM"] = time

display(tabela.head())

In [ ]:
col_TEAM = tabela.pop("TEAM")
tabela.insert(1, "TEAM", col_TEAM)

display(tabela.head())

In [ ]:
tabela.insert(0, "ID", -1)
for linha in tabela.index:
    tabela.loc[linha, "ID"] = linha

In [ ]:
display(tabela.head())

In [ ]:
colunas_numericas = ["RATING", "ACS", "K/D", "ADR", "KPR", "APR", "FKPR", "FDPR"]
colunas_porcentagem = ["KAST", "HS%"]
tabela[colunas_numericas] = tabela[colunas_numericas].astype(float)
for coluna in colunas_porcentagem:
    tabela[coluna] = tabela[coluna].str.replace("%", "").astype(float) / 100

display(tabela.info())

In [ ]:
display(tabela.head())

#### \>

#### Analizing Stats

In [ ]:
import plotly.express as ply

In [ ]:
# Stats/team
kast_por_time = tabela.groupby("TEAM")[["KAST"]].mean()
kast_por_time = kast_por_time.sort_values("KAST", ascending=False)
kast_por_time = kast_por_time[["KAST"]] * 100
display(kast_por_time)

In [ ]:
kast_por_time[:12].plot(kind="barh", figsize=(15, 5))

#### \>

In [ ]:
# get camp
camp = navegador.find_element(By.CLASS_NAME, 'wf-title').text
display(camp)

#### \<game cathalog>

In [ ]:
# Acessando matches
for item in nav_itens:
    if 'matches' in item.get_attribute("href"):
        item.click()
        link = navegador.current_url
        break

In [ ]:
navegador.find_element(By.PARTIAL_LINK_TEXT, 'Completed').click()

In [ ]:
print(len(navegador.find_element(By.CLASS_NAME, 'wf-subnav') \
             .find_elements(By.PARTIAL_LINK_TEXT, 'All')))

In [ ]:
print(link)

In [ ]:
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

In [ ]:
matches = navegador.find_elements(By.CLASS_NAME, 'match-item')
pri_win = navegador.window_handles[0]
for match in matches:
    ActionChains(navegador).key_down(Keys.CONTROL).click(match).key_up(Keys.CONTROL).perform()
    sleep(.5)
    seg_win = navegador.window_handles[1]
    navegador.switch_to.window(seg_win)

    display(navegador.find_element(By.CLASS_NAME, 'match-header-vs').text)
    navegador.close()
    sleep(.5)
    navegador.switch_to.window(pri_win)

## </game>

In [ ]:
# buscando times:
times = navegador.find_elements(By.CLASS_NAME, 'wf-title-med')
for item in times:
    display(item.text)

In [ ]:
# Buscando placar:
placar = navegador.find_elements(By.CLASS_NAME, 'js-spoiler')
try:
    display(placar[0].text)
except: 
    display("NaN")

In [ ]:
# Buscando pickban:
pickban = navegador.find_element(By.CLASS_NAME, 'match-header-note').text
display(pickban)
new_pb = pickban.split('; ')
lista_maps = []
for item in new_pb:
    if "pick" in item:
        lista_maps.append(item.split(" ")[2])
lista_maps.append(new_pb[6].split(" ")[0])
display(lista_maps)

In [ ]:
# Trocando mapas:
mapas = navegador.find_elements(By.CLASS_NAME, 'js-map-switch')
dict_maps = {}
for i, item in enumerate(lista_maps):
    dict_maps[item] = mapas[i+1]

map_counter = 0
dict_maps[lista_maps[map_counter]].click()

In [ ]:
# Encontrar agentes (Partidas completas)
maps = navegador.find_elements(By.CLASS_NAME, 'vm-stats-game')
compositions = []
for map in maps:
    if map.get_attribute('class').split(' ')[1] != '':
        continue
    
    agentes = map.find_elements(By.CLASS_NAME, 'mod-agents')
    compositionA = []
    compositionB = []
    for i, agente in enumerate(agentes):
        linha = agente.find_elements(By.TAG_NAME, 'img')
        try:
            if i < 5:
                compositionA.append(linha[0].get_attribute('title'))
            else:
                compositionB.append(linha[0].get_attribute('title'))
        except:
            break
    compositions.append((compositionA, compositionB))
display(compositions)

In [ ]:
# Encontrar agentes (Partidas incompletas)
maps = navegador.find_elements(By.CLASS_NAME, 'vm-stats-game')
for map in maps:
    if map.get_attribute('class').split(' ')[1] == '':
        continue
    
    agentes = map.find_elements(By.CLASS_NAME, 'mod-agents')
    compositionA = []
    compositionB = []
    for i, agente in enumerate(agentes):
        linha = agente.find_elements(By.TAG_NAME, 'img')
        try:
            if i < 5:
                compositionA.append(linha[0].get_attribute('title'))
            else:
                compositionB.append(linha[0].get_attribute('title'))
        except:
            break
    composition = (compositionA, compositionB)
    break

display(composition)

In [ ]:
# Buscando rounds:
maps_headers = navegador.find_elements(By.CLASS_NAME, 'vm-stats-game')
rounds = []
for header in maps_headers:
    if header.find_elements(By.CLASS_NAME, 'map') == []:
        continue
    if header.find_element(By.CLASS_NAME, 'map').find_element(By.TAG_NAME, 'span').text.split("\n")[0] == lista_maps[map_counter]:
        rounds = header.find_elements(By.CLASS_NAME, 'vlr-rounds-row-col')
        break

print(lista_maps[map_counter])

for round in rounds:
    display(round.get_attribute('title'))

In [ ]:
try:
    first = rounds[1]
    if first.get_attribute('title') == '0-1':
        if first.find_element(By.CLASS_NAME, 'mod-win').get_attribute("class").split(" ")[2] == 'mod-t':
            atq = 'B'
        else:
            atq = 'A'
    else:
        if first.find_element(By.CLASS_NAME, 'mod-win').get_attribute("class").split(" ")[2] == 'mod-t':
            atq = 'A'
        else:
            atq = 'B'
except:
    print('algo deu errado')

# Desconbrindo quem começa atacando
print(atq)

In [ ]:
# Monta o round_sequence

start = ['0', '0']
round_sequence = ''
stop = True
OT = False
for round in rounds[1:]:
    round = round.get_attribute('title').split('-')
    if round == ['']:
        round_sequence += 'X'
    elif start[0] != round[0]:
        round_sequence += "A"
        start[0] = round[0]
    elif start[1] != round[1]:
        round_sequence += "B"
        start[1] = round[1]
    
    if '13' in start and stop:
        break
    elif abs(int(start[0]) - int(start[1])) == 2 and OT:
        break
    elif start == ['12', '12']:
        OT = True
        stop = False



display(round_sequence)

# Testes


In [ ]:
game = {
    'camp': 'VCT 2026: China Kickoff', 
    'times': ['sen', 'g2'], 
    'mapas': [
        {'id': 'corrode', 
         'atk': 'B', 
         'composicoes': [['Yoru', 'Waylay', 'Omen', 'Viper', 'Fade'], 
                         ['Sova', 'Kayo', 'Omen', 'Viper', 'Waylay']], 
         'rounds': 'BAAAABAAABBBXAAAABAA', 
         'win': 'A'
         }, 
         {'id': 'haven', 
          'atk': 'B', 
          'composicoes': [['Cypher', 'Omen', 'Sova', 'Yoru', 'Neon'], 
                          ['Killjoy', 'Neon', 'Viper', 'Sova', 'Omen']], 
          'rounds': 'AABBBBAABBBAXBBBBABB', 
          'win': 'B'
         }, 
         {'id': 'abyss', 
          'atk': 'B', 
          'composicoes': [['Waylay', 'Cypher', 'Yoru', 'Astra', 'Sova'], 
                          ['Sova', 'Tejo', 'Waylay', 'Neon', 'Astra']], 
          'rounds': 'BBAAAAABBAAAXAABBBAABBBA', 
          'win': 'A'
          }
        ], 
        'pickban': {
            'Abans': ['split', 'bind'], 
            'Bbans': ['pearl', 'breeze'], 
            'Apicks': ['haven'], 
            'Bpicks': ['corrode'], 
            'decider': 'abyss'}, 
        'winner': 'A'}